In [1]:
%autosave 0

Autosave disabled


In [2]:
%pip install grpcio==1.42.0 tensorflow-serving-api==2.7.0
%pip install keras-image-helper

  Using cached grpcio-1.42.0.tar.gz (21.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached tensorflow_serving_api-2.7.0-py2.py3-none-any.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of tensorboard to determine which version is compatible with other requirements. This could take a while.
  Using cached tensorflow-2.20.0-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
  Using cached tensorflow-2.19.1-cp311-cp311-win_amd64.whl.metadata (4.1 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached numpy-2.1.3-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached tensorflow_io_gcs_filesystem-0.31.0-cp311-cp311-win_amd64.whl.metadata (14 kB)
  Usin

ERROR: Cannot install grpcio==1.42.0, tensorflow, tensorflow-intel and tensorflow-serving-api==2.7.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np

import grpc
import tensorflow as tf
from tensorflow_serving.apis import prediction_service_pb2_grpc, predict_pb2
from tensorflow import make_tensor_proto, make_ndarray

print("Imports successful!")

Imports successful!


In [5]:
host = "localhost:8500" #"127.0.0.1:8500" 
channel = grpc.insecure_channel(host)
prediction_service_stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

In [6]:
from keras_image_helper import create_preprocessor

In [46]:
url = "https://bit.ly/49Dxq1l"
preprocessor = create_preprocessor('xception', target_size=(299, 299))
X = preprocessor.from_url(url)

In [ ]:
# Initialize the request
predict_request = predict_pb2.PredictRequest()
predict_request.model_spec.name = "clothing-model"
predict_request.model_spec.signature_name = "serving_default"
predict_request.inputs["input_layer_16"].CopyFrom(make_tensor_proto(X))

Success! Data loaded into request.


In [49]:
predict_response = prediction_service_stub.Predict(predict_request, timeout=20.0) 

In [50]:
pred = predict_response.outputs['output_0'].float_val

In [31]:
category_names = [
    "Belts",
    "Briefs",
    "Casual Shoes",
    "Flip Flops",
    "Handbags",
    "Heels",
    "Kurtas",
    "Sandals",
    "Shirts",
    "Sports Shoes",
    "Sunglasses",
    "Tops",
    "Tshirts",
    "Wallets",
    "Watches",
]

In [51]:
dict(zip(category_names, pred))

{'Belts': -2.30871844291687,
 'Briefs': -3.335125684738159,
 'Casual Shoes': -1.5288379192352295,
 'Flip Flops': -4.0694355964660645,
 'Handbags': -8.292472839355469,
 'Heels': -4.67220401763916,
 'Kurtas': -3.2223899364471436,
 'Sandals': -4.204179763793945,
 'Shirts': 0.47840842604637146,
 'Sports Shoes': -4.389585494995117,
 'Sunglasses': -5.942244052886963,
 'Tops': 2.704115152359009,
 'Tshirts': 9.847935676574707,
 'Wallets': -6.93977689743042,
 'Watches': -6.351983547210693}